# Data Injestion with LakeFlow Connect

Listing files in our volume

In [0]:
SELECT 1;

In [0]:
LIST '/Volumes/workspace/default/raw_files'

### Creating Single Delta Table from raw files 

In [0]:
CREATE OR REPLACE TABLE customers_bronze AS 
SELECT *
FROM read_files(
  '/Volumes/workspace/default/raw_files/',
  format => 'csv',
  header => true
);

### Setting up Production level streaming delta table with 1 Hour Schedule time

In [0]:
SELECT * FROM customers_bronze


In [0]:
CREATE OR REFRESH STREAMING TABLE customers_bronze_streaming
SCHEDULE EVERY 1 HOUR
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/workspace/default/raw_files/',
  format => 'csv',
  header => true
);

In [0]:
DESCRIBE TABLE EXTENDED customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_streaming

Refreshing Column After adding new files to folder.

In [0]:
REFRESH STREAMING TABLE customers_bronze_streaming

In [0]:
SELECT * FROM customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_streaming

Result of running the DESCRIBE HISTORY AFTER AN HOUR to see real demo.

In [0]:
SELECT * FROM customers_bronze_streaming

In [0]:
DESCRIBE HISTORY customers_bronze_history

### Injestion Data with Meta Data

In [0]:
SELECT
*, 
-- Last data source file modification time
_metadata.file_name AS source_file,
-- Ingest data source file name
current_timestamp() as ingestion_time
-- Ingestion timestamp
FROM read_files (
"/Volumes/workspace/default/raw_files");

It's working as expected , so lets create final bronze table from this. 

In [0]:
DROP TABLE IF EXISTS customers_bronze_with_meta_data;

CREATE TABLE customers_bronze_with_meta_data AS
SELECT
*, 
-- Last data source file modification time
_metadata.file_name AS source_file,
-- Ingest data source file name
current_timestamp() as ingestion_time
-- Ingestion timestamp
FROM read_files (
"/Volumes/workspace/default/raw_files");

SELECT * FROM customers_bronze_with_meta_data;

In [0]:
SELECT 
source_file,
COUNT(*) AS total
FROM customers_bronze_with_meta_data
GROUP BY source_file
ORDER BY source_file;